<a href="https://colab.research.google.com/github/fidlarsyn/Introduction-Machine-Learning-with-python/blob/main/BAB_6_Algorithm_Chains_and_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fondasi: Membangun Pipeline Dasar**
Pipeline memungkinkan kita menggabungkan beberapa langkah pemrosesan data (seperti transformasi) dan tahap akhir berupa estimator ke dalam satu objek tunggal. Hal ini menyederhanakan kode dan memastikan urutan transformasi selalu konsisten.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

# 1. Persiapan Data (Reproducibility adalah kunci)
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

# 2. Inisialisasi Pipeline dengan daftar tupel (nama, objek)
# Format: ('nama_langkah', objek_estimator)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

# 3. Melatih Pipeline
# StandardScaler.fit() dijalankan pada X_train, kemudian ditransformasikan,
# lalu hasilnya digunakan untuk SVC.fit()
pipe.fit(X_train, y_train)

# 4. Evaluasi
print("Skor data uji: {:.2f}".format(pipe.score(X_test, y_test)))

# **Integrasi Pipeline dalam Grid Search**
Menggunakan Pipeline di dalam GridSearchCV bukan sekadar masalah kerapian kode, melainkan keharusan teknis untuk menghindari data leakage.
Dengan memasukkan Pipeline ke dalam GridSearchCV, scaling hanya dilakukan pada training fold di setiap iterasi cross-validation.

Pesan Pakar (Warning): Jika Anda melakukan scaling (seperti StandardScaler) pada seluruh dataset sebelum melakukan cross-validation, informasi dari test fold (seperti nilai mean dan std global) akan bocor ke dalam proses pelatihan. Ini menyebabkan estimasi akurasi yang terlalu optimis.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Mendefinisikan parameter grid dengan sintaks '__' (double underscore)
# Format: 'nama_langkah__nama_parameter'
# Ini disebut sebagai "Nesting parameters"
param_grid = {
    'svm__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1, 10, 100]
}

# GridSearchCV membungkus pipeline sebagai estimator utama
grid = GridSearchCV(pipe, param_grid=param_grid, cv=5)
grid.fit(X_train, y_train)

print("Akurasi cross-validation terbaik: {:.2f}".format(grid.best_score_))
print("Skor data uji: {:.2f}".format(grid.score(X_test, y_test)))
print("Parameter terbaik: {}".format(grid.best_params_))

# **Pembuatan Pipeline Instan dengan make_pipeline**
Fungsi make_pipeline adalah convenience function yang secara otomatis menamai setiap langkah berdasarkan nama kelasnya (dalam huruf kecil).

In [ ]:
from sklearn.pipeline import make_pipeline

# Versi standar (Pipeline)
pipe_long = Pipeline([("scaler", StandardScaler()), ("svm", SVC())])

# Versi ringkas (make_pipeline)
pipe_short = make_pipeline(StandardScaler(), SVC())

# Nuansa Teknis: Jika ada dua langkah dari kelas yang sama,
# make_pipeline akan menambahkan angka di belakangnya.
pipe_duplicate = make_pipeline(StandardScaler(), StandardScaler())
print("Nama langkah otomatis:\n", pipe_duplicate.steps)
# Output akan menunjukkan: ('standardscaler-1', ...), ('standardscaler-2', ...)

# **Mengakses Atribut Langkah-Langkah dalam Pipeline**
Untuk memeriksa atribut internal model setelah dilatih (seperti koefisien regresi atau komponen PCA), kita menggunakan atribut named_steps yang bersifat seperti dictionary.

In [ ]:
from sklearn.decomposition import PCA

# Membangun pipeline dengan PCA
pipe = make_pipeline(StandardScaler(), PCA(n_components=2), StandardScaler())
pipe.fit(cancer.data)

# Mengambil objek 'pca' dari langkah pipeline untuk melihat komponennya
components = pipe.named_steps['pca'].components_
print("Bentuk komponen PCA: {}".format(components.shape))

# **Akses Atribut pada Pipeline dalam Grid Search**
Setelah GridSearchCV selesai, model terbaik dapat diakses melalui best_estimator_. Dari sana, kita bisa mengekstraksi atribut spesifik dari langkah di dalamnya.

In [ ]:
# Menggunakan objek 'grid' dari Section 2
# Mengakses SVC dari estimator terbaik hasil Grid Search
best_svc = grid.best_estimator_.named_steps['svm']

print("Parameter C terbaik pada model final:", best_svc.C)

# **Grid Search pada Tahap Preprocessing dan Pemilihan Model**
Teknik tingkat lanjut ini memungkinkan kita mencari kombinasi terbaik antara teknik preprocessing dan model klasifikasi dalam satu pencarian tunggal menggunakan list of dictionaries.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler

# Definisi pipeline dengan nama langkah placeholder
pipe = Pipeline([('preprocessing', StandardScaler()), ('classifier', SVC())])

param_grid = [
    # Dictionary 1: Mencoba SVC dengan berbagai scaler
    {
        'classifier': [SVC()],
        'preprocessing': [StandardScaler(), MinMaxScaler()],
        'classifier__gamma': [0.001, 0.01, 0.1, 1, 10, 100],
        'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100]
    },

    # Dictionary 2: Mencoba RandomForest
    # Pro-tip: RandomForest tidak sensitif terhadap skala data,
    # sehingga kita set preprocessing ke [None] untuk efisiensi komputasi.
    {
        'classifier': [RandomForestClassifier(n_estimators=100)],
        'preprocessing': [None],
        'classifier__max_features': [1, 2, 3]
    }
]

# Jalankan Grid Search dengan random_state tetap untuk reproduksibilitas
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

print("Akurasi terbaik: {:.2f}".format(grid.best_score_))
print("Kombinasi parameter terbaik:\n", grid.best_params_)